Exploring Spatial abundances, first part until lab 12/03 only based on composition and using q05

In [ ]:
#Setup and Rendering Configuration

import scanpy as sc
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy.stats import gmean

# Optimize Scanpy for Jupyter inline rendering
sc.settings.verbosity = 3
sc.set_figure_params(dpi=100, facecolor='white')

SPATIAL_FILE = "/path/to/data/Cell2location/spatial_mapping/spatial_mapped_with_posterior.h5ad"
OUT_DIR = "/path/to/data/Cell2location/spatial_mapping/exploration"
os.makedirs(OUT_DIR, exist_ok=True)
sc.settings.figdir = OUT_DIR

In [ ]:
#Data Loading & CLR Transformation

print(f"Loading mapped spatial atlas: {SPATIAL_FILE}")
adata_vis = sc.read_h5ad(SPATIAL_FILE)

print("Extracting q05 posteriors & calculating compositional fractions...")
abundances = adata_vis.obsm['q05_cell_abundance_w_sf'].copy()
abundances.columns = [col.replace('q05cell_abundance_w_sf_', '') for col in abundances.columns]
alpha = abundances.div(abundances.sum(axis=1), axis=0)

print("Applying Centered Log-Ratio (CLR) transformation...")
eps = 1e-6
alpha_clipped = np.clip(alpha, eps, 1 - eps)
geo_means = gmean(alpha_clipped.values, axis=1, keepdims=True)
clr_abundances = np.log(alpha_clipped.values / geo_means)

print("Initializing compositional AnnData object...")
adata_comp = sc.AnnData(X=clr_abundances, obs=adata_vis.obs.copy())
adata_comp.var_names = abundances.columns
adata_comp.obsm['spatial'] = adata_vis.obsm['spatial'].copy()

if 'spatial' in adata_vis.uns:
    adata_comp.uns['spatial'] = adata_vis.uns['spatial'].copy()

In [ ]:
# Dimensionality Reduction & Threshold Justification

print("Running PCA...")
sc.tl.pca(adata_comp, svd_solver='arpack')

# Visualizing the elbow plot directly in the notebook to justify thresholds
sc.pl.pca_variance_ratio(adata_comp, n_pcs=50)

In [ ]:
#Graph Computation & Clustering

print("Computing neighbors and UMAP on baseline CLR space (n_pcs=10)...")
sc.pp.neighbors(adata_comp, n_neighbors=15, n_pcs=10)
sc.tl.umap(adata_comp)

print("Executing compositional Leiden clustering...")
sc.tl.leiden(adata_comp, resolution=0.5, key_added='spatial_niche', flavor="igraph", n_iterations=2, directed=False)

In [ ]:
# Interactive Validation Visualizations
adata_comp = sc.read_h5ad('/path/to/data/Cell2location/spatial_mapping/exploration_mean/spatial_compositional_space_mean.h5ad')
# Batch check using the engineered slide_id
batch_cols = ['study', 'batch_source', 'slide_id']
sc.pl.umap(adata_comp, color=batch_cols, ncols=2, save="_baseline_batch_check_metadata_change.png")

# Biology check
bio_cols = ['condition', 'diagnosis', 'tissue_status', 'isup_grade', 'zone']
sc.pl.umap(adata_comp, color=bio_cols, ncols=2, save="_baseline_biology_check_metadata_change.png")

# Spatial niches
sc.pl.umap(adata_comp, color=['spatial_niche'], save="_baseline_niches_umap_metadata_change.png")

In [ ]:
# Saving output

out_file = os.path.join(OUT_DIR, "spatial_compositional_space.h5ad")
adata_comp.write(out_file)
print(f"✅ Compositional AnnData successfully saved to: {out_file}")

try to fix the plot wordings:

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc

# Load compositional space
adata_comp = sc.read_h5ad('/path/to/data/Cell2location/spatial_mapping/exploration_mean/spatial_compositional_space_mean.h5ad')

# Set standard layout style and typography natively
plt.style.use('seaborn-v0_8-white' if 'seaborn-v0_8-white' in plt.style.available else 'default')
plt.rcParams.update({
    'font.size': 9,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'figure.dpi': 300
})

# Minimal scanpy initialization
sc.set_figure_params(vector_friendly=True)

# ==========================================
# 1. Batch Check (Handling the large slide_id legend)
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

# Plot panels (letting Scanpy generate the raw labels internally)
sc.pl.umap(adata_comp, color='study', ax=axes[0], show=False)
handles0, labels0 = axes[0].get_legend_handles_labels()
axes[0].legend(handles0, labels0, bbox_to_anchor=(1.05, 1), loc="upper left", frameon=False)

sc.pl.umap(adata_comp, color='batch_source', ax=axes[1], show=False)
handles1, labels1 = axes[1].get_legend_handles_labels()
axes[1].legend(handles1, labels1, bbox_to_anchor=(1.05, 1), loc="upper left", ncol=2, frameon=False)

sc.pl.umap(adata_comp, color='slide_id', ax=axes[2], show=False)
handles2, labels2 = axes[2].get_legend_handles_labels()
# Give slide_id 3 columns and tiny text size so it sits comfortably beside the panel
axes[2].legend(handles2, labels2, bbox_to_anchor=(1.05, 1), loc="upper left", ncol=3, fontsize=7, frameon=False)

# Safely drop the unutilized window panel
fig.delaxes(axes[3])

# Padding optimization
plt.tight_layout()
fig.subplots_adjust(wspace=0.8, hspace=0.3)  # Increased wspace slightly for the 3-column slide_id



# ==========================================
# 2. Biology Check (Fixing cut-off text labels)
# ==========================================
bio_cols = ['condition', 'diagnosis', 'tissue_status', 'isup_grade', 'zone']

fig_bio, axes_bio = plt.subplots(3, 2, figsize=(14, 15))
axes_bio = axes_bio.flatten()

for i, col in enumerate(bio_cols):
    sc.pl.umap(adata_comp, color=col, ax=axes_bio[i], show=False)
    h, l = axes_bio[i].get_legend_handles_labels()
    # Check for crowded categories to dynamically widen columns
    num_cols = 2 if col in ['diagnosis', 'isup_grade'] else 1
    axes_bio[i].legend(h, l, bbox_to_anchor=(1.05, 1), loc="upper left", ncol=num_cols, frameon=False)

# Drop the 6th unused subplot panel layout
fig_bio.delaxes(axes_bio[5])

plt.tight_layout()
fig_bio.subplots_adjust(wspace=0.6, hspace=0.3)



# ==========================================
# 3. Spatial Niches
# ==========================================
fig_niche, ax_niche = plt.subplots(figsize=(7, 5))
sc.pl.umap(adata_comp, color='spatial_niche', ax=ax_niche, show=False)
hn, ln = ax_niche.get_legend_handles_labels()
ax_niche.legend(hn, ln, bbox_to_anchor=(1.05, 1), loc="upper left", frameon=False)

plt.tight_layout()


After lab meeting 12/02, we see that there might be batch effects and we have to check this:
(spatial_compositional_space_mean.h5ad was made with this notebook: 11_explore_spatial_abundances_mean.ipynb). Is the separation between Erickson and Hu in UMAP space is a technical batch effect or real biology

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.stats import entropy, gmean

# ── PATHS ─────────────────────────────────────────────────────────────
OUT_DIR = "/path/to/data/Cell2location/spatial_mapping/exploration_mean"
os.makedirs(OUT_DIR, exist_ok=True)

# ── CELL 1: Load ──────────────────────────────────────────────────────
# adata_comp IS already the mean-based compositional AnnData
adata_mean = sc.read_h5ad(
    "/path/to/data/Cell2location/spatial_mapping/exploration_mean/spatial_compositional_space_mean.h5ad"
)
print("adata_mean shape:", adata_mean.shape)
print("obs columns:", adata_mean.obs.columns.tolist())
print("studies:", adata_mean.obs['study'].unique())

# Load adata_vis ONLY to get raw non-CLR abundances for heatmap
adata_vis = sc.read_h5ad(
    "/path/to/data/Cell2location/spatial_mapping/spatial_mapped_with_posterior.h5ad"
)
mean_abundances = adata_vis.obsm['means_cell_abundance_w_sf'].copy()
mean_abundances.columns = [col.replace('meanscell_abundance_w_sf_', '')
                            for col in mean_abundances.columns]
print("✅ Raw mean abundances loaded:", mean_abundances.shape)
print("Cell type name check:", mean_abundances.columns[:5].tolist())

In [ ]:

# ── CELL 2: Split by study and run independent Leiden ─────────────────
study_adatas = {}

for study in adata_mean.obs['study'].unique():
    print(f"\nProcessing {study}...")
    adata_sub = adata_mean[adata_mean.obs['study'] == study].copy()
    print(f"  Spots: {adata_sub.n_obs}")
    
    sc.tl.pca(adata_sub, svd_solver='arpack')
    sc.pp.neighbors(adata_sub, n_neighbors=15, n_pcs=10)
    sc.tl.umap(adata_sub)
    
    for res in [0.3, 0.4, 0.5, 0.6]:
        sc.tl.leiden(adata_sub, resolution=res,
                     key_added=f'leiden_res{res}',
                     flavor="igraph", n_iterations=2, directed=False)
        n = adata_sub.obs[f'leiden_res{res}'].nunique()
        print(f"  res={res}: {n} clusters")
    
    study_adatas[study] = adata_sub

In [ ]:

# ── CELL 3: Entropy function ──────────────────────────────────────────
def calculate_entropy_scores(adata_sub, cluster_key, groupby_col):
    results = []
    clusters = adata_sub.obs[cluster_key].unique()
    global_n_groups = adata_sub.obs[groupby_col].nunique()
    H_max = np.log2(global_n_groups) if global_n_groups > 1 else 1
    
    for cluster in sorted(clusters):
        mask = adata_sub.obs[cluster_key] == cluster
        cluster_obs = adata_sub.obs.loc[mask, groupby_col]
        counts = cluster_obs.value_counts()
        proportions = counts / counts.sum()
        H = entropy(proportions, base=2)
        H_norm = H / H_max
        
        results.append({
            'cluster': cluster,
            'n_spots': mask.sum(),
            'n_groups_in_cluster': len(counts),
            'n_groups_global': global_n_groups,
            'entropy_raw': H,
            'entropy_normalised': H_norm
        })
    return pd.DataFrame(results).sort_values('cluster')


In [ ]:
# ── CELL 4: Choose resolutions and run entropy ────────────────────────
# Adjust these after inspecting cluster counts above
chosen_res = {
    'Erickson_2022': 'leiden_res0.5',
    'Hu_2025': 'leiden_res0.5'
}

entropy_results = {}
for study, adata_sub in study_adatas.items():
    cluster_key = chosen_res[study]
    print(f"\n=== {study} ===")
    
    ent_patient = calculate_entropy_scores(adata_sub, cluster_key, 'slide_id')
    ent_patient['study'] = study
    ent_patient['entropy_type'] = 'by_patient'
    
    ent_study = calculate_entropy_scores(adata_sub, cluster_key, 'study')
    ent_study['study'] = study
    ent_study['entropy_type'] = 'by_study'
    
    entropy_results[study] = {
        'by_patient': ent_patient,
        'by_study': ent_study
    }
    print(ent_patient[['cluster', 'n_spots', 'entropy_raw',
                        'entropy_normalised']].to_string())


In [ ]:
# ── CELL 5: Combined Leiden + entropy ────────────────────────────────
# Explicitly recompute graph for mathematical consistency
# Uses identical parameters to per-study graphs in Cell 2
sc.tl.pca(adata_mean, svd_solver='arpack')
sc.pp.neighbors(adata_mean, n_neighbors=15, n_pcs=10)
for res in [0.3, 0.4, 0.5]:
    key = f'leiden_combined_res{res}'
    if key not in adata_mean.obs.columns:  # skip if already computed
        sc.tl.leiden(adata_mean, resolution=res,
                     key_added=key,
                     flavor="igraph", n_iterations=2, directed=False)
    n = adata_mean.obs[key].nunique()
    print(f"Combined res={res}: {n} clusters")

combined_cluster_key = 'leiden_combined_res0.5'  # adjust if needed

ent_combined_study = calculate_entropy_scores(
    adata_mean, combined_cluster_key, 'study')
ent_combined_patient = calculate_entropy_scores(
    adata_mean, combined_cluster_key, 'slide_id')

print("\nCombined entropy by study:")
print(ent_combined_study[['cluster', 'n_spots',
                           'entropy_normalised']].to_string())


In [ ]:
# ── CELL 6: Transfer per-study labels back to adata_mean ─────────────
for study, adata_sub in study_adatas.items():
    cluster_key = chosen_res[study]
    mask = adata_mean.obs['study'] == study
    adata_mean.obs.loc[mask, cluster_key] = adata_sub.obs[cluster_key].values

print(adata_mean.obs[['study', 'leiden_res0.5']].value_counts().sort_index())

In [ ]:
# ── CELL 7: Visualizations ────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Entropy by patient per study
ax = axes[0, 0]
for study, res_dict in entropy_results.items():
    df = res_dict['by_patient']
    x = [f"{study}\nC{c}" for c in df['cluster']]
    ax.bar(x, df['entropy_normalised'], alpha=0.7, label=study)
ax.axhline(y=0.7, color='red', linestyle='--', label='Threshold (0.7)')
ax.set_ylabel('Normalised Entropy')
ax.set_title('Per-cluster Patient Mixing\nHigh = biological, Low = batch')
ax.set_ylim(0, 1)
ax.legend()
ax.tick_params(axis='x', rotation=45)

# Plot 2: Study composition per cluster
ax = axes[0, 1]
study_per_cluster = adata_mean.obs.groupby(
    [combined_cluster_key, 'study']).size().unstack(fill_value=0)
study_per_cluster_norm = study_per_cluster.div(
    study_per_cluster.sum(axis=1), axis=0)
study_per_cluster_norm.plot(kind='bar', stacked=True, ax=ax,
                             colormap='Set1', alpha=0.8)
ax.set_xlabel('Cluster')
ax.set_ylabel('Proportion of spots')
ax.set_title('Study Composition per Cluster\n50/50 = no batch effect')
ax.legend(title='Study')
ax.tick_params(axis='x', rotation=0)

# Plot 3: Combined entropy by study
ax = axes[1, 0]
ax.bar(ent_combined_study['cluster'],
       ent_combined_study['entropy_normalised'],
       color='steelblue', alpha=0.7)
ax.axhline(y=0.7, color='red', linestyle='--')
ax.set_xlabel('Cluster')
ax.set_ylabel('Normalised Entropy')
ax.set_title('Combined Space: Study Mixing')
ax.set_ylim(0, 1)

# Plot 4: Combined entropy by patient (by slide_id- so by sample)
ax = axes[1, 1]
ax.bar(ent_combined_patient['cluster'],
       ent_combined_patient['entropy_normalised'],
       color='darkorange', alpha=0.7)
ax.axhline(y=0.7, color='red', linestyle='--')
ax.set_xlabel('Cluster')
ax.set_ylabel('Normalised Entropy')
ax.set_title('Combined Space: Patient Mixing')
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'batch_entropy_analysis.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:

# ── CELL 8: Clustermap ────────────────────────────────────────────────
cluster_proportions = pd.DataFrame(
    mean_abundances.values,
    index=adata_mean.obs.index,
    columns=mean_abundances.columns
)

erickson_mask = adata_mean.obs['study'] == 'Erickson_2022'
hu_mask = adata_mean.obs['study'] == 'Hu_2025'

erickson_means = (cluster_proportions[erickson_mask]
                  .groupby(adata_mean.obs.loc[erickson_mask,
                           chosen_res['Erickson_2022']]).mean())
hu_means = (cluster_proportions[hu_mask]
            .groupby(adata_mean.obs.loc[hu_mask,
                     chosen_res['Hu_2025']]).mean())

erickson_means.index = 'Erickson_' + erickson_means.index.astype(str)
hu_means.index = 'Hu_' + hu_means.index.astype(str)

combined_means = pd.concat([erickson_means, hu_means])
combined_means_norm = combined_means.div(
    combined_means.sum(axis=1), axis=0)

top_ct = combined_means_norm.std().nlargest(20).index

g = sns.clustermap(
    combined_means_norm[top_ct].T,
    cmap='viridis',
    method='ward',
    standard_scale=1,
    figsize=(12, 10),
    xticklabels=True,
    yticklabels=True
)
g.fig.suptitle(
    'Hierarchical Clustering: Erickson vs Hu Independent Niches\n'
    'Interleaved = biological signal | Separated = batch effect',
    y=1.02
)
g.savefig(os.path.join(OUT_DIR, 'niche_clustermap_batch_check.png'),
          dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# UMAP of per-study independent clustering
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Erickson independent clustering
sc.pl.umap(study_adatas['Erickson_2022'], 
           color='leiden_res0.5',
           title='Erickson_2022 — Independent Leiden (res=0.5)',
           ax=axes[0], show=False)

# Plot 2: Hu independent clustering  
sc.pl.umap(study_adatas['Hu_2025'],
           color='leiden_res0.5',
           title='Hu_2025 — Independent Leiden (res=0.5)',
           ax=axes[1], show=False)

# Plot 3: Combined clustering
sc.pl.umap(adata_mean,
           color='leiden_combined_res0.5',
           title='Combined — Leiden (res=0.5)',
           ax=axes[2], show=False)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'umap_clustering_overview.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Erickson: which slides contribute to which cluster
print("=== Erickson_2022: Cluster → Slide breakdown ===")
erickson_ct = pd.crosstab(
    study_adatas['Erickson_2022'].obs['leiden_res0.5'],
    study_adatas['Erickson_2022'].obs['slide_id'],
    normalize='index'  # normalise by row so each cluster sums to 1
)
print(erickson_ct.round(3))

# Hu: which slides contribute to which cluster
print("\n=== Hu_2025: Cluster → Slide breakdown ===")
hu_ct = pd.crosstab(
    study_adatas['Hu_2025'].obs['leiden_res0.5'],
    study_adatas['Hu_2025'].obs['slide_id'],
    normalize='index'
)
print(hu_ct.round(3))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import os
import seaborn as sns

# --- TASK 1: METADATA CORRELATION WITH CLUSTERMAP BRANCHES ---
print("Executing Task 1: Dendrogram Branch Analysis...")

# Define the branches exactly as they appear in the clustermap output
epithelial_immune_branch = [
    'Erickson_6', 'Erickson_8', 'Erickson_0', 'Erickson_7',
    'Hu_3', 'Hu_5', 'Erickson_2', 'Hu_4'
]

stromal_branch = [
    'Erickson_4', 'Hu_1', 'Hu_2', 'Hu_7', 'Erickson_1',
    'Hu_0', 'Hu_6', 'Hu_8', 'Erickson_3', 'Erickson_5'
]

def map_niche_to_branch(row):
    study = str(row['study'])
    cluster = str(row['leiden_res0.5'])
    
    if 'Erickson' in study:
        label = f"Erickson_{cluster}"
    else:
        label = f"Hu_{cluster}"
    
    if label in epithelial_immune_branch:
        return 'Epithelial/Immune Branch'
    elif label in stromal_branch:
        return 'Stromal Branch'
    else:
        return 'Unassigned'

adata_mean.obs['dendrogram_branch'] = adata_mean.obs.apply(
    map_niche_to_branch, axis=1)

# Verify assignment counts
print(adata_mean.obs['dendrogram_branch'].value_counts())

# Visualize metadata distribution across branches
metadata_cols = ['tissue_status', 'diagnosis', 'zone']
valid_cols = [col for col in metadata_cols if col in adata_mean.obs.columns]

if valid_cols:
    fig, axes = plt.subplots(1, len(valid_cols), figsize=(6 * len(valid_cols), 6))
    # Handle single axis case
    if len(valid_cols) == 1: axes = [axes]
        
    for ax, col in zip(axes, valid_cols):
        # Calculate cross-tabulation (normalize by index to get proportions)
        ct = pd.crosstab(adata_mean.obs['dendrogram_branch'], adata_mean.obs[col], normalize='index')
        ct.plot(kind='bar', stacked=True, ax=ax, colormap='Set2', alpha=0.9)
        
        ax.set_title(f'Branch vs {col}')
        ax.set_ylabel('Proportion of Spots')
        ax.set_xlabel('Clustermap Branch')
        ax.tick_params(axis='x', rotation=0)
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title=col)

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'dendrogram_metadata_correlation.png'), dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Check what samples have images
print("Samples with spatial images:")
print(list(adata_mean.uns['spatial'].keys()))

# Plot H&E slides — Erickson vs Hu comparison
# Pick one representative sample from each study
erickson_samples = (adata_mean.obs[adata_mean.obs['study'] == 'Erickson_2022']
                    ['slide_id'].unique())
hu_samples = (adata_mean.obs[adata_mean.obs['study'] == 'Hu_2025']
              ['slide_id'].unique())

print("Erickson samples:", erickson_samples)
print("Hu samples:", hu_samples)

# Plot 2 from each study — no color, just H&E
fig, axes = plt.subplots(2, 2, figsize=(16, 16))

samples_to_plot = [
    (erickson_samples[0], 'Erickson_2022', axes[0,0]),
    (erickson_samples[1], 'Erickson_2022', axes[0,1]),
    (hu_samples[0],       'Hu_2025',       axes[1,0]),
    (hu_samples[1],       'Hu_2025',       axes[1,1]),
]

for sample_id, study, ax in samples_to_plot:
    mask = adata_mean.obs['slide_id'] == sample_id
    adata_sample = adata_mean[mask].copy()
    
    # Plot with no color — just tissue image
    sc.pl.spatial(adata_sample,
                  color=None,
                  library_id=sample_id,
                  show=False,
                  ax=ax,
                  title=f'{study}\n{sample_id}',
                  spot_size=1.5)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'HE_slides_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print(adata_mean.uns.keys())
print(adata_vis.uns.keys())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

# ── PATH DEFINITIONS ──────────────────────────────────────────────────
ERICKSON_P1_PATH = "/path/to/data/Erickson_2022_Data/svw96g68dv-1/Histological_images/Patient_1/Visium/"
ERICKSON_P2_PATH = "/path/to/data/Erickson_2022_Data/svw96g68dv-1/Histological_images/Patient_2/"
HU_BASE_PATH = "/path/to/data/Junyi_2025/visium/"

def get_image_path(slide_id):
    if slide_id.startswith('Patient_1_'):
        suffix = slide_id.replace('Patient_1_', '')
        return os.path.join(ERICKSON_P1_PATH, f"{suffix}.jpg")
    elif slide_id.startswith('Patient_2_'):
        suffix = slide_id.replace('Patient_2_', '')
        return os.path.join(ERICKSON_P2_PATH, f"{suffix}.jpg")
    else:
        return os.path.join(HU_BASE_PATH, slide_id, 
                           'spatial', 'tissue_hires_image.png')

# ── VERIFY ALL PATHS ──────────────────────────────────────────────────
print("Verifying image paths:")
for slide_id in sorted(adata_mean.obs['slide_id'].unique()):
    path = get_image_path(slide_id)
    exists = os.path.exists(path)
    print(f"{'✅' if exists else '❌'} {slide_id}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib
import os
from PIL import Image
import numpy as np

# ── DISABLE PIL SIZE LIMIT ────────────────────────────────────────────
# Required for high-resolution histology images
Image.MAX_IMAGE_PIXELS = None  # removes the decompression bomb limit

def load_image(path):
    """Load image safely, resizing if too large"""
    if path.endswith('.jpg') or path.endswith('.jpeg'):
        # Use PIL for jpg — allows size limit override
        img = Image.open(path)
        # Downsample to max 4000px on longest side for display
        img.thumbnail((4000, 4000), Image.LANCZOS)
        return np.array(img)
    else:
        # png files are fine with matplotlib directly
        return mpimg.imread(path)

# ── 2x2 COMPARISON ───────────────────────────────────────────────────
comparison_slides = [
    ('Patient_1_H1_2', 'Erickson_2022'),
    ('Patient_2_H3_1', 'Erickson_2022'),
    ('HP09_CZ_ST',     'Hu_2025'),
    ('HP05_PZ_ST',     'Hu_2025'),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 16))
axes = axes.flatten()

for ax, (slide_id, study) in zip(axes, comparison_slides):
    path = get_image_path(slide_id)
    print(f"Loading {slide_id} from {path}...")
    
    if os.path.exists(path):
        try:
            img = load_image(path)
            ax.imshow(img)
            ax.set_title(f'{study}\n{slide_id}',
                         fontsize=12, fontweight='bold')
            print(f"  ✅ Loaded — shape: {img.shape}")
        except Exception as e:
            ax.text(0.5, 0.5, f'Error loading:\n{str(e)[:50]}',
                    ha='center', va='center',
                    transform=ax.transAxes, color='red', fontsize=8)
            print(f"  ❌ Error: {e}")
    else:
        ax.text(0.5, 0.5, f'File not found:\n{slide_id}',
                ha='center', va='center',
                transform=ax.transAxes, color='red')
        print(f"  ❌ Not found: {path}")
    
    ax.axis('off')

plt.suptitle('H&E Histology: Erickson_2022 vs Hu_2025',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'HE_comparison_2x2.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- TASK 3: GLOBAL COMPOSITION BARPLOT ---
print("\nExecuting Task 3: Global Cell Type Proportions per Slide...")

# Use the raw mean abundances directly
cell_props = mean_abundances.copy()
cell_props['slide_id'] = adata_mean.obs['slide_id'].values
cell_props['study'] = adata_mean.obs['study'].values

# Calculate the mean proportion of each cell type per slide
sample_means = cell_props.groupby(['study', 'slide_id']).mean().reset_index()

# Normalize so each slide sums to 1.0 (100%)
numeric_cols = [col for col in sample_means.columns if col not in ['study', 'slide_id']]
sample_means[numeric_cols] = sample_means[numeric_cols].div(sample_means[numeric_cols].sum(axis=1), axis=0)

# Sort by study so Erickson and Hu are grouped together
sample_means = sample_means.sort_values('study')
sample_means.set_index('slide_id', inplace=True)

epithelial = [c for c in numeric_cols if 'epithelial' in c]
stromal    = [c for c in numeric_cols if any(x in c for x in
              ['smooth_muscle', 'fibroblast', 'endothelial'])]
immune     = [c for c in numeric_cols if c not in epithelial + stromal]
order      = epithelial + stromal + immune

plot_data = sample_means[order]
# Verify sums to 1
print("Sum check:", plot_data.sum(axis=1).describe())
# Plot the stacked bar chart
fig, ax = plt.subplots(figsize=(20, 8))
plot_data.plot(kind='bar', stacked=True, ax=ax, colormap='tab20', width=0.85)

# SAFER — preserves sort order:
study_order = sample_means['study'].tolist()
n_erickson = study_order.count('Erickson_2022')
n_hu = study_order.count('Hu_2025')
ax.axvline(x=n_erickson - 0.5, color='black', linewidth=2.5, linestyle='--')
ax.text(n_erickson/2, 1.02, 'Erickson_2022', 
        ha='center', transform=ax.get_xaxis_transform(), fontsize=14, fontweight='bold')
ax.text(n_erickson + n_hu/2, 1.02, 'Hu_2025',
        ha='center', transform=ax.get_xaxis_transform(), fontsize=14, fontweight='bold')
        
ax.set_title('Global Cell Type Composition per Physical Slide', fontsize=16)
ax.set_ylabel('Mean Proportion', fontsize=12)
ax.set_xlabel('Slide ID', fontsize=12)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'global_composition_per_slide.png'), dpi=150, bbox_inches='tight')
plt.show()

print("✅ Analysis Complete.")

Does these biological differences translate into the umap ? i.e. do the high epithelial samples cluster together and the low also together ? Can that explain the batch effect ?

In [ ]:
# Check exact cell type names
print(adata_mean.var_names.tolist())

strip prefixes

In [ ]:
# Check current var_names
print("Current var_names prefix example:", adata_mean.var_names[0])

# Strip the prefix
adata_mean.var_names = [v.replace('meanscell_abundance_w_sf_', '') 
                         for v in adata_mean.var_names]

# Verify
print("Fixed var_names:", adata_mean.var_names.tolist())

In [ ]:
print("mean_abundances columns:", mean_abundances.columns[:3].tolist())

# Fix if needed
mean_abundances.columns = [col.replace('meanscell_abundance_w_sf_', '') 
                            for col in mean_abundances.columns]

print("Fixed:", mean_abundances.columns.tolist())

In [ ]:
#Define lineage groups:
epithelial_celltypes = [
    'epithelial_basal', 'epithelial_club', 'epithelial_hillock',
    'epithelial_luminal', 'epithelial_neuroendocrine'
]
stromal_celltypes = [
    'smooth_muscle', 'fibroblast', 'endothelial'
]
immune_celltypes = [
    'b', 'dendritic', 'macrophage_group_1', 'macrophage_group_2',
    'mast', 'monocyte', 'natural_killer_cd56_bright', 'natural_killer_cd56_dim',
    'plasma', 't_cd4_naive_central_memory', 't_cd4_tissue_resident_memory',
    't_cd8_cytotoxic', 't_cd8_tissue_resident_memory', 't_regulatory'
]

# Verify all 22 are covered
all_assigned = set(epithelial_celltypes + stromal_celltypes + immune_celltypes)

print(f"Total assigned: {len(all_assigned)}/22")
print(f"Missing: {[ct for ct in adata_mean.var_names if ct not in all_assigned]}")

In [ ]:
#Calculate lineage scores:

# Use raw mean abundances normalised to proportions (non-CLR, interpretable)
raw_props = mean_abundances.div(mean_abundances.sum(axis=1), axis=0)

# Add scores to adata_mean obs
adata_mean.obs['epithelial_score'] = raw_props[epithelial_celltypes].sum(axis=1).values
adata_mean.obs['stromal_score']    = raw_props[stromal_celltypes].sum(axis=1).values
adata_mean.obs['immune_score']     = raw_props[immune_celltypes].sum(axis=1).values

# Sanity check
print("Mean lineage scores per study:")
print(adata_mean.obs.groupby('study')[
    ['epithelial_score', 'stromal_score', 'immune_score']
].mean().round(3))


In [ ]:
# Key UMAP plot: lineage scores vs study identity

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Row 1: lineage scores — do they spatially explain the UMAP structure?
sc.pl.umap(adata_mean, color='epithelial_score',
           cmap='Reds', ax=axes[0,0], show=False,
           title='Epithelial Score\n(luminal + basal + club + hillock + neuroendocrine)')
sc.pl.umap(adata_mean, color='stromal_score',
           cmap='Blues', ax=axes[0,1], show=False,
           title='Stromal Score\n(smooth muscle + fibroblast + endothelial)')
sc.pl.umap(adata_mean, color='immune_score',
           cmap='Greens', ax=axes[0,2], show=False,
           title='Immune Score\n(T/B/NK/macrophage/dendritic/mast/plasma)')

# Row 2: study and tissue metadata on SAME UMAP coordinates for direct comparison
sc.pl.umap(adata_mean, color='study',
           ax=axes[1,0], show=False,
           title='Study Identity\n← compare with epithelial score above')
sc.pl.umap(adata_mean, color='tissue_status',
           ax=axes[1,1], show=False,
           title='Tissue Status\n(Malignant / Normal)')
sc.pl.umap(adata_mean, color='diagnosis',
           ax=axes[1,2], show=False,
           title='Diagnosis')

plt.suptitle(
    "Does epithelial/stromal composition explain the Erickson vs Hu UMAP separation?\n"
    "If red epithelial region overlaps with Erickson (blue) → separation is biological, not batch",
    fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'umap_biology_explains_separation.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#Cell 5 — Quantitative confirmation: Spearman correlations

from scipy.stats import spearmanr

umap_coords = adata_mean.obsm['X_umap']
adata_mean.obs['UMAP1'] = umap_coords[:, 0]
adata_mean.obs['UMAP2'] = umap_coords[:, 1]

print("Spearman correlation of lineage scores with UMAP axes:")
print(f"{'Score':<20} {'UMAP1 r':>10} {'UMAP2 r':>10}")
print("-" * 42)
for score in ['epithelial_score', 'stromal_score', 'immune_score']:
    r1, p1 = spearmanr(adata_mean.obs['UMAP1'], adata_mean.obs[score])
    r2, p2 = spearmanr(adata_mean.obs['UMAP2'], adata_mean.obs[score])
    print(f"{score:<20} {r1:>10.3f} {r2:>10.3f}")

In [ ]:
#Cell 6 — Violin plot: lineage score distributions per study


fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, score, color in zip(
        axes,
        ['epithelial_score', 'stromal_score', 'immune_score'],
        ['#d62728', '#1f77b4', '#2ca02c']):

    data_e = adata_mean.obs.loc[adata_mean.obs['study'] == 'Erickson_2022', score]
    data_h = adata_mean.obs.loc[adata_mean.obs['study'] == 'Hu_2025', score]

    parts = ax.violinplot([data_e, data_h], positions=[0, 1], showmedians=True)
    for pc in parts['bodies']:
        pc.set_facecolor(color)
        pc.set_alpha(0.7)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Erickson_2022\n(PCa)', 'Hu_2025\n(Normal/BLCA)'], fontsize=10)
    ax.set_title(score.replace('_score', '').capitalize() + ' Score', fontsize=12)
    ax.set_ylabel('Proportion of total cell abundance')

    # Add mean values as text
    ax.text(0, data_e.mean() + 0.01, f'μ={data_e.mean():.3f}', ha='center', fontsize=9)
    ax.text(1, data_h.mean() + 0.01, f'μ={data_h.mean():.3f}', ha='center', fontsize=9)

plt.suptitle(
    'Lineage Score Distribution by Study\n'
    'Confirms biological basis of UMAP separation',
    fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'lineage_scores_violin_by_study.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#Cell 7 — Summary scatter: epithelial vs stromal per spot, colored by study


# Scatter of epithelial vs stromal score per spot, colored by study
# If the two studies occupy different regions of this plot → biological separation confirmed

fig, ax = plt.subplots(figsize=(8, 6))

colors = {'Erickson_2022': '#1f77b4', 'Hu_2025': '#ff7f0e'}
for study, color in colors.items():
    mask = adata_mean.obs['study'] == study
    ax.scatter(
        adata_mean.obs.loc[mask, 'epithelial_score'],
        adata_mean.obs.loc[mask, 'stromal_score'],
        c=color, alpha=0.05, s=1, label=study, rasterized=True
    )

ax.set_xlabel('Epithelial Score (proportion)', fontsize=12)
ax.set_ylabel('Stromal Score (proportion)', fontsize=12)
ax.set_title(
    'Epithelial vs Stromal Score per Spot\n'
    'Separated clouds = biological difference, not batch effect',
    fontsize=11)

# Fix legend marker size for visibility
legend = ax.legend(markerscale=15, fontsize=11)
for lh in legend.legend_handles:
    lh.set_alpha(1)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'epithelial_vs_stromal_scatter.png'),
            dpi=150, bbox_inches='tight')
plt.show()

Conclusion: The UMAP separation is biological. Erickson spots are more immune-enriched, Hu spots are more stromal-enriched